# Fact Knowledge Layer — Colab (T4) Run

Run **all** cells in order. On a free T4 GPU this extracts all 5 PDFs and
generates the four evaluation cases in roughly 20-40 minutes. Results are
downloaded as a zip at the end. Nothing here needs a paid API.

> Note: after the first run, an **on-disk cache** (`data/fact_cache/`) makes
> any repeat run near-instant, so you can re-upload a *new* PDF cheaply.


### 0. Connect to a GPU (runtime > Change runtime type > T4 GPU)

In [ ]:
import os
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)
!nvidia-smi | head -3


### 1. Install the project into this environment

In [ ]:
import sys, subprocess, pathlib
ROOT = pathlib.Path("/content/fact-knowledge-layer")
if not ROOT.exists():
    !git clone https://github.com/<YOUR_USERNAME>/fact-knowledge-layer.git /content/fact-knowledge-layer
else:
    print("already cloned")
%cd /content/fact-knowledge-layer
!pip install -q pdfplumber rapidfuzz python-multipart requests pytesseract pillow 2>&1 | tail -1
!apt-get -qq install -y tesseract-ocr >/dev/null 2>&1 && echo "tesseract-ocr installed"


### 2. Install & start Ollama on this Colab VM (no sudo needed)

In [ ]:
import subprocess, time, pathlib, urllib.request
# Install ollama binary (no sudo on Colab)
p = pathlib.Path("/usr/local/bin/ollama")
if not p.exists():
    url = "https://ollama.com/download/ollama-linux-amd64.tgz"
    print("downloading ollama...")
    !curl -fsSL -o /tmp/ollama.tgz {url}
    !tar -xzf /tmp/ollama.tgz -C /usr/local
    print("installed")
!ollama --version


In [ ]:
import subprocess, os, time
# Start the ollama server in the background
env = dict(os.environ)
env["OLLAMA_HOST"] = "127.0.0.1:11434"
# Keep the large model resident so we don't reload between calls
env["OLLAMA_KEEP_ALIVE"] = "1h"

log = open("/tmp/ollama.log", "w")
proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=log)
time.sleep(4)
print("ollama server pid:", proc.pid)
!curl -s --max-time 5 http://127.0.0.1:11434/api/tags | head -c 200


### 3. Pull the model

`qwen3:8b` gives noticeably better JSON/quote fidelity than `:4b` and still fits the T4's 16 GB.

In [ ]:
!ollama pull qwen3:8b
!ollama list | grep qwen3


### 4. Point the code at YOUR pdfs

Either edit the list below to your own PDF paths, or upload new ones with the file-upload widget and add their filenames.

In [ ]:
from google.colab import files
# OPTIONAL: upload any PDFs you want to test the layer with.
uploaded = files.upload()  # skip if you already cloned the repo with data/


### 5. Extract facts from all PDFs (one LLM call per page)

This is the only slow step. Everything after this is instant.

In [ ]:
import os, sys, time, json
sys.path.insert(0, "/content/fact-knowledge-layer")
os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = "qwen3:8b"

from src.extraction.pdf_extractor import PDFExtractor
from src.extraction.llm_client import LLMClient

llm = LLMClient()   # provider='ollama', model=qwen3:8b, OLLAMA_HOST=127.0.0.1:11434
extractor = PDFExtractor(llm_client=llm, max_workers=2)

base = pathlib.Path("/content/fact-knowledge-layer/data")
pdfs = sorted(base.rglob("*.pdf"))
pdfs = [p for p in pdfs if "uploads" not in str(p)]
print("found", len(pdfs), "pdfs")
for p in pdfs:
    t = time.time()
    facts = extractor.extract_facts(str(p), source_document=p.name)
    print(f"  {p.name}: {len(facts)} facts in {time.time()-t:.0f}s", flush=True)


### 6. Run the four required cases

Loads facts from cache (instant), runs the comparator that classifies each cross-document pair.

In [ ]:
from src.comparison.comparator import FactComparator
from src.storage.database import Database

db = Database(db_path="data/facts_colab.db")
f_all = []
for p in [x for x in pdfs if "uploads" not in str(x)]:
    facts = extractor.extract_facts(str(p), source_document=p.name)
    f_all.extend(facts)
    for f in facts:
        db.save_fact(f)
print("total facts:", len(f_all))

comparator = FactComparator(llm_client=llm)
comparisons = comparator.compare_facts(f_all)
print("comparisons:", len(comparisons))
json.dump([c.model_dump() for c in comparisons],
          open("comparisons.json","w"), indent=2, default=str)
print(open("comparisons.json").read()[:200])


### 7. Inspect the four cases

Each case must show the *source evidence* and the model's *reasoning* (the `explanation` field). Run this to print them.

In [ ]:
import json
comps = json.load(open("comparisons.json"))
def show(rel):
    hits = [c for c in comps if c.get("relationship") == rel]
    print(f"--- {rel}: {len(hits)} ---")
    for c in hits[:3]:
        print("  EXPL:", c.get("explanation","")[:220])
        print("  CONF:", c.get("confidence"))
        print()
show("corroborates")
show("contradicts")
show("reconciled")


### 8. Package everything

Downloads a zip with facts, comparisons, and stats so you can attach it to the assignment or use it in the demo video.

In [ ]:
import json
from src.storage.database import Database
db = Database("data/facts_colab.db")
facts = db.get_all_facts()
json.dump(facts, open("facts_export.json","w"), indent=2, default=str)
!zip -q -r results.zip facts_export.json comparisons.json data/fact_cache data/*.pdf
print("zip created (~%d KB)" % (os.path.getsize("results.zip")//1024))
from google.colab import files
files.download("results.zip")
print("downloaded. Done!")


### Done

- Extract ran once; results live in `data/fact_cache/`.
- Uploading a **new** PDF and calling `extract_facts` again works the same way (cache misses only for the new file).
- Scanned/image-only PDFs are handled automatically: pages with no selectable
  text are OCR'd with Tesseract (installed above) and their facts are tagged
  `context["text_source"] = "ocr"`.